# C6: Pierwsza wtyczka QGIS — szkielet i interfejs
### Programowanie w GIS — Cwiczenia

---

> **Kurs:** Programowanie w GIS (QGIS)  
> **Cwiczenie:** C6  
> **Czas:** ~90 minut  

---

### Cele cwiczenia

Po ukonczeniu tego cwiczenia bedziesz potrafil/a:

- recznie stworzyc minimalna strukture plikow wtyczki QGIS,
- napisac wymagane metody `__init__`, `initGui`, `unload` i `run`,
- zainstalowac i uruchomic wtyczke w QGIS,
- zaprojektowac okno dialogowe w Qt Designer,
- podlaczyc widgety do kodu przez sygnaly i sloty,
- korzystac z Plugin Reloader do szybkiego testowania.

---


### Przygotowanie

Przed rozpoczeciem cwiczenia:

1. Zainstaluj **Plugin Reloader** przez Menedzera wtyczek.
2. Znajdz folder wtyczek QGIS na swoim komputerze.
   Sprawdz sciezke w Konsoli Pythona:

```python
import qgis.utils
print(qgis.utils.plugin_paths)
```

3. Otworz dowolny edytor tekstu (VS Code, Notepad++, PyCharm).
4. Wczytaj do projektu warstwe panstw Natural Earth.


---
<a id='s1'></a>

## 1. Tworzenie struktury plikow od podstaw

Zamiast uzywac Plugin Builder, stworzymy wtyczke recznie.
Dzieki temu zobaczysz dokladnie co jest niezbedne i dlaczego.

### Zadanie 1.1 — Stworz folder wtyczki

W folderze wtyczek QGIS stworz nowy folder o nazwie:

```
moja_pierwsza_wtyczka
```

Nazwa folderu to jednoczesnie nazwa modulu Pythona.
Musi byc w snake_case (male litery, podkreslenia zamiast spacji).

### Zadanie 1.2 — Stworz puste pliki

Wewnatrz folderu `moja_pierwsza_wtyczka/` stworz trzy puste pliki:

```
moja_pierwsza_wtyczka/
    __init__.py
    metadata.txt
    moja_pierwsza_wtyczka.py
```

To jest **minimalna struktura** wymagana przez QGIS.
Bez zadnego z tych plikow wtyczka sie nie zaladuje.


---
<a id='s2'></a>

## 2. Plik __init__.py — classFactory

Plik `__init__.py` to punkt wejscia do pakietu.
QGIS szuka w nim funkcji `classFactory` i wywoluje ja przy ladowaniu wtyczki.

### Zadanie 2.1 — Napisz __init__.py

Otworz plik `__init__.py` w edytorze i wpisz dokladnie:

```python
def classFactory(iface):
    from .moja_pierwsza_wtyczka import MojaPierwszaWtyczka
    return MojaPierwszaWtyczka(iface)
```

Zapisz plik.

Zwroc uwage na:

- kropke przed nazwa modulu (`.moja_pierwsza_wtyczka`) — import wzgledny,
- nazwe klasy `MojaPierwszaWtyczka` — za chwile ja napiszemy,
- argument `iface` — QGIS przekazuje tu obiekt interfejsu.


**Pytanie 2.1** — Co by sie stalo gdybysmy napisali
`from moja_pierwsza_wtyczka import MojaPierwszaWtyczka`
zamiast `from .moja_pierwsza_wtyczka import MojaPierwszaWtyczka`
(bez kropki)?
Na czyim komputerze mogloby to przestac dzialac i dlaczego?


---
<a id='s3'></a>

## 3. Plik metadata.txt

QGIS czyta `metadata.txt` zanim zaladuje kod wtyczki.
Na podstawie tego pliku decyduje czy wtyczka jest kompatybilna
i wyswietla informacje w Menedzerze wtyczek.

### Zadanie 3.1 — Napisz metadata.txt

Otworz plik `metadata.txt` i wpisz:

```ini
[general]
name=Moja Pierwsza Wtyczka
qgisMinimumVersion=3.0
description=Pierwsza wtyczka napisana recznie na cwiczeniach
version=0.1
author=Twoje Imie i Nazwisko
email=twoj@email.com
about=Wtyczka demonstracyjna stworzona podczas cwiczen z programowania w GIS.
experimental=True
```

Uzupelnij `author` i `email` swoimi danymi. Zapisz plik.

Opcjonalnie: `icon=icon.png`, pozwala dodać jej ikonkę. `icon.png` musi znajdować się w tym samym folderze.


---
<a id='s4'></a>

## 4. Glowna klasa wtyczki

Teraz napiszemy serce wtyczki — klase z trzema wymaganymi metodami.

### Zadanie 4.1 — Napisz minimalny szkielet

Otworz `moja_pierwsza_wtyczka.py` i wpisz:

```python
from qgis.PyQt.QtWidgets import QAction
from qgis.PyQt.QtGui import QIcon


class MojaPierwszaWtyczka:

    def __init__(self, iface):
        self.iface   = iface
        self.actions = []
        self.menu    = '&Moja Wtyczka'
        self.dlg     = None

    def initGui(self):
        action = QAction('Uruchom Moja Wtyczke', self.iface.mainWindow())
        action.triggered.connect(self.run)
        self.iface.addToolBarIcon(action)
        self.iface.addPluginToMenu(self.menu, action)
        self.actions.append(action)

    def unload(self):
        for action in self.actions:
            self.iface.removePluginMenu(self.menu, action)
            self.iface.removeToolBarIcon(action)

    def run(self):
        self.iface.messageBar().pushInfo(
            'Moja Wtyczka', 'Dziala! To jest moja pierwsza wtyczka.'
        )
```

Zapisz plik.


**Pytanie 4.1** — Dlaczego w `unload()` iterujemy po `self.actions`
zamiast trzymac jedna zmienna `self.action`?
Kiedy lista `self.actions` moglaby miec wiecej niz jeden element?

### Zadanie 4.2 — Zrozum przepływ sterowania

Przenalizuj kolejnosc wywolan i wypelnij schemat:

```
QGIS startuje
    |
    +--> import moja_pierwsza_wtyczka     (co wykonuje?)
    |    ___________________________________
    |
    +--> classFactory(iface)              (co zwraca?)
    |    ___________________________________
    |
    +--> obiekt.__init__(iface)           (co inicjalizuje?)
    |    ___________________________________
    |
    +--> obiekt.initGui()                 (co dodaje do QGIS?)
         ___________________________________
```


In [ ]:
# Uzupelnij schemat (wpisz co dzieje sie w kazdym kroku):


---
<a id='s5'></a>

## 5. Instalacja i pierwsze uruchomienie

### Zadanie 5.1 — Wlacz wtyczke w Menedzerze

1. W QGIS przejdz do `Wtyczki -> Zarządzaj wtyczkami`
2. W zakladce **Ustawienia** zaznacz opcje
   **Pokazuj rowniez wtyczki eksperymentalne**
3. Przejdz do zakladki **Zainstalowane**
4. Znajdz `Moja Pierwsza Wtyczka` i zaznacz checkbox
5. Kliknij **Zamknij**

W pasku narzedzi powinna pojawic sie nowa ikona
(lub tekst, jeśli dodalismy wlasnej ikony).

### Zadanie 5.2 — Przetestuj dzialanie

1. Kliknij ikone wtyczki lub
   `Wtyczki -> Moja Wtyczka -> Uruchom Moja Wtyczke`
2. Na gorze obszaru mapy powinien pojawic sie zielony pasek komunikatu

Jesli wtyczka sie nie pojawia — sprawdz:

```python
# [QGIS] W Konsoli Pythona:
import qgis.utils
# Czy folder jest widoczny?
print(qgis.utils.plugin_paths)
# Jakie wtyczki sa zaladowane?
print(list(qgis.utils.plugins.keys()))
```
**UWAGA:**
Bledy  znajdziesz w:
`Widok -> Panele -> Log Messages -> Python`


**Pytanie 5.1** — Co sie stanie jesli w `unload()` zapomnisz
usunac akcje z paska narzedzi i ponownie wlaczysz wtyczke?
Sprawdz to eksperymentalnie: wykomentuj linie w `unload()`,
wylacz i wlacz wtyczke w Menedzerze i zaobserwuj efekt.


In [ ]:
# Twoja odpowiedz i opis eksperymentu:


---
<a id='s6'></a>

## 6. Qt Designer — projektowanie dialogu

Teraz dodamy prawdziwe okno dialogowe zamiast tylko komunikatu.

W folderze wtyczki dodamy dwa nowe pliki (nie dodawaj ich ręcznie):

```
moja_pierwsza_wtyczka/
    __init__.py
    metadata.txt
    moja_pierwsza_wtyczka.py
    dialog.py                 <- NOWY
    dialog_base.ui            <- NOWY (stworzy Qt Designer)
```

### Zadanie 6.1 — Zaprojektuj dialog w Qt Designer

Uruchom Qt Designer i otworz nowy plik jako **Dialog with Buttons Bottom**.

**Na ćwiczeniach zapoznamy się dokładnie z właściwościami QtDesignera**

Zaprojektuj okno z nastepujacymi elementami:

| Widget | objectName | Ustawienia |
|---|---|---|
| QLabel | `wynik` | Tekst: `Wynik: ` |
| QPushButton | `analizuj` | Tekst: `Analizuj` |

Ustaw nazwy w opcji edycji właściwości.


i zapisz jako `dialog_base.ui` w folderze wtyczki.


**Pytanie 6.1** — Czym jest plik `.ui` pod wzgledem technicznym?
Otworz go w edytorze tekstu (nie Qt Designer) i sprawdz jego zawartosc.
Jakiego formatu danych uzywa? Znajdz w nim objectName widgetu `lblWarstwa`.


In [ ]:
# Twoja odpowiedz:


---
<a id='s7'></a>

## 7. Podlaczenie GUI do kodu

### Zadanie 7.1 — Dodaj "knowerter" okna

Dodaj plik do konwersji. W folderze dodaj plik `dialog.py`, a w nim

```
import os
from qgis.PyQt import uic, QtWidgets

# uic.loadUiType() czyta plik .ui i zwraca:
# - FORM_CLASS: klase opisujaca interfejs (widgety, layouty)
# - _: klase bazowa (zazwyczaj QDialog — ignorujemy)
FORM_CLASS, _ = uic.loadUiType(
    os.path.join(os.path.dirname(__file__), 'dialog_base.ui')
)

class MojDialog(QtWidgets.QDialog, FORM_CLASS):

    def __init__(self, parent=None):
        super().__init__(parent)
        self.setupUi(self)  # <-- tu wstrzykuje widgety z pliku .ui
```

### Zadanie 7.2 — Zaktualizuj glowna klase

Zastap zawartosc `moja_pierwsza_wtyczka.py` na:

```python
from qgis.PyQt.QtWidgets import QAction
from qgis.PyQt.QtGui import QIcon
from .dialog import MojDialog # DODAJEMY IMPORT KLASY!

class MojaPierwszaWtyczka:

    def __init__(self, iface):
        self.iface   = iface
        self.actions = []
        self.menu    = '&Moja Wtyczka'
        self.window     = None

    def initGui(self):
        action = QAction('Uruchom Moja Wtyczke', self.iface.mainWindow())
        action.triggered.connect(self.run)
        self.iface.addToolBarIcon(action)
        self.iface.addPluginToMenu(self.menu, action)
        self.actions.append(action)

    def unload(self):
        for action in self.actions:
            self.iface.removePluginMenu(self.menu, action)
            self.iface.removeToolBarIcon(action)

    def run(self):
        if self.window is None:
            self.window = MojDialog()  # przygotowuje pustą klasę MojDialog (musi nazywac sie jak w pliku dialog.py)
        self.window.show()  # pokazujemy okno
```

Zapisz plik.


**Pytanie 7.1** — Dlaczego tworzymy dialog przez
`if self.dlg is None: self.dlg = MojDialog()`
zamiast po prostu `self.dlg = MojDialog()` w metodzie `run()`?


---
<a id='s8'></a>

## 8. Plugin Reloader — szybkie testowanie

### Zadanie 8.1 — Skonfiguruj Plugin Reloader

1. Upewnij sie ze Plugin Reloader jest zainstalowany i wlaczony
2. `Wtyczki -> Plugin Reloader -> Configure`
3. Z listy wybierz `moja_pierwsza_wtyczka`
4. Kliknij OK

Od teraz po kazdej zmianie kodu nacisnij przycik Plugin Reloader zmiast restartowac QGIS.

### Zadanie 8.2 — Przetestuj cykl wprowadzania zmian

Wykonaj ponizszy cykl:

```
1. Zmien cos w kodzie (np. tresc komunikatu w messageBar)
2. Zapisz plik
3. Uruchum PluginRealoade
4. Uruchom wtyczke i sprawdz zmiane
```

### Zadanie 8.3 — Diagnozowanie bledow

Celowo wprowadz blad skladni do `moja_pierwsza_wtyczka.py`
(np. usun dwukropek po `def run()`) i zaktualizuj wtyczke.

Sprawdz gdzie pojawia sie informacja o bledzie:

- Konsola Pythona QGIS
- `Widok -> Panele -> Log Messages -> Python`

Napraw blad i potwierdz ze wtyczka znowu dziala.


**Pytanie 8.1** — Opisz co widzisz w log Messages
gdy wtyczka ma blad skladni.
Jakie informacje sa pomocne przy diagnostyce?


In [ ]:
# Twoja odpowiedz:


---
<a id='s9'></a>

## 9. Pierwsza funkcjonalność wtyczki

### Zadanie 9.1 — Nowa funkcjonalność i działanie sygnału i slotu

Do metody `run()` dodaj tę linijkę kodu

`self.window.[nazwa_przycisku].clicked.connect(self.licz_obiekty)`

Do pliku dodaj nową metodę:

```
    def licz_obiekty(self):
        warstwa = self.iface.activeLayer()

        if warstwa is None:
            self.window.[nazwa_label].setText('Nie wybrano warstwy.')
            return

        n = warstwa.featureCount()
        self.window.[nazwa_label].setText(f'Warstwa "{warstwa.name()}" ma {n} obiektów.')
```

### Zadanie 9.2 — Rozbuduj wtyczke

Rozbuduj swoja wtyczke o

**Przycisk eksportu do CSV**

Dodaj przycisk `Eksportuj do CSV` ktory zapisuje
tabele atrybutow aktywnej warstwy do pliku CSV.
Uzyj `QFileDialog.getSaveFileName()` do wyboru sciezki.


In [1]:
# Wklej tutaj finalny kod swojej rozbudowanej wtyczki:

# [QGIS]


---

## Podsumowanie C6

W tym cwiczeniu:

- stworzyłes/stworzylas minimalna strukture wtyczki od podstaw (3 pliki),
- napisal/a wymagane metody: `classFactory`, `__init__`, `initGui`, `unload`, `run`,
- zrozumiales/zrozumialas przepływ sterowania przy starcie QGIS,
- zaprojektowales/zaprojektowałas dialog w Qt Designer z `QgsMapLayerComboBox`,
- podlaczyłes/podlaczylas sygnaly i sloty,
- skonfigurował/a Plugin Reloader do szybkiego testowania.

---
